# **Load Needed Libraries and Data**

In [ ]:
!pip install sweetviz
!pip install ydata-profiling
!pip install pandera
!pip install great_expectations
!pip install cerberus
!pip install faker

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sweetviz as sv
from ydata_profiling import ProfileReport
import pandera as pa
from pandera import Column, Index, DataFrameSchema, Check
import great_expectations as gx
import cerberus
from cerberus import Validator
import os
from IPython.display import display

/tmp/ipykernel_6021/2453663814.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

In [ ]:
# Load some csv files for mearge
orders   = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
payments  = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv"))
reviews   = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))

In [ ]:
df = orders.merge(customers, on="customer_id") \
           .merge(payments,  on="order_id") \
           .merge(reviews,   on="order_id")

print(f"Dataset shape: {df.shape}")
print(df.head())

In [ ]:
print(type(df))
print(df.shape)
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
(103677, 22)
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp'],
      dtype='object')


Other Dataset:

In [ ]:
new_path = kagglehub.dataset_download("sahilprajapati143/retail-analysis-large-dataset")

print("Path to dataset files:", new_path)

Using Colab cache for faster access to the 'retail-analysis-large-dataset' dataset.
Path to dataset files: /kaggle/input/retail-analysis-large-dataset


In [ ]:
retail_data = pd.read_csv(os.path.join(new_path, "new_retail_data.csv"))

In [ ]:
retail_data.shape

(302010, 30)

In [ ]:
retail_data.columns

Index(['Transaction_ID', 'Customer_ID', 'Name', 'Email', 'Phone', 'Address',
       'City', 'State', 'Zipcode', 'Country', 'Age', 'Gender', 'Income',
       'Customer_Segment', 'Date', 'Year', 'Month', 'Time', 'Total_Purchases',
       'Amount', 'Total_Amount', 'Product_Category', 'Product_Brand',
       'Product_Type', 'Feedback', 'Shipping_Method', 'Payment_Method',
       'Order_Status', 'Ratings', 'products'],
      dtype='object')

In [ ]:
retail_data.head()

,Transaction_ID,Customer_ID,Name,Email,Phone,Address,City,State,Zipcode,Country,...,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,products
0,8691788.0,37249.0,Michelle Harrington,Ebony39@gmail.com,1.414787e+09,3959 Amanda Burgs,Dortmund,Berlin,77985.0,Germany,...,324.086270,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5.0,Cycling shorts
1,2174773.0,69749.0,Kelsey Hill,Mark36@gmail.com,6.852900e+09,82072 Dawn Centers,Nottingham,England,99071.0,UK,...,806.707815,Electronics,Samsung,Tablet,Excellent,Standard,Credit Card,Processing,4.0,Lenovo Tab
2,6679610.0,30192.0,Scott Jensen,Shane85@gmail.com,8.362160e+09,4133 Young Canyon,Geelong,New South Wales,75929.0,Australia,...,1063.432799,Books,Penguin Books,Children's,Average,Same-Day,Credit Card,Processing,2.0,Sports equipment
3,7232460.0,62101.0,Joseph Miller,Mary34@gmail.com,2.776752e+09,8148 Thomas Creek Suite 100,Edmonton,Ontario,88420.0,Canada,...,2466.854021,Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,4.0,Utility knife
4,4983775.0,27901.0,Debra Coleman,Charles30@gmail.com,9.098268e+09,5813 Lori Ports Suite 269,Bristol,England,48704.0,UK,...,248.553049,Grocery,Nestle,Chocolate,Bad,Standard,Cash,Shipped,1.0,Chocolate cookies


New Dataset to Be Used:

In [ ]:
Extra_Columns = retail_data[["Email", "Phone", "Address", "Age", "Gender", "Income", "Customer_Segment"]].copy()

In [ ]:
Extra_Columns

,Email,Phone,Address,Age,Gender,Income,Customer_Segment
0,Ebony39@gmail.com,1.414787e+09,3959 Amanda Burgs,21.0,Male,Low,Regular
1,Mark36@gmail.com,6.852900e+09,82072 Dawn Centers,19.0,Female,Low,Premium
2,Shane85@gmail.com,8.362160e+09,4133 Young Canyon,48.0,Male,Low,Regular
3,Mary34@gmail.com,2.776752e+09,8148 Thomas Creek Suite 100,56.0,Male,High,Premium
4,Charles30@gmail.com,9.098268e+09,5813 Lori Ports Suite 269,22.0,Male,Low,Premium
...,...,...,...,...,...,...,...
302005,Courtney60@gmail.com,7.466354e+09,389 Todd Path Apt. 159,31.0,Male,Medium,Regular
302006,Jennifer71@gmail.com,5.754305e+09,52809 Mark Forges,35.0,Female,Low,New
302007,Christopher100@gmail.com,9.382530e+09,407 Aaron Crossing Suite 495,41.0,Male,Low,Premium
302008,Rebecca65@gmail.com,9.373222e+09,3204 Baird Port,41.0,Male,Medium,New


In [ ]:
Extra_Columns = Extra_Columns.rename(columns={
    "Email": "customer_email",
    "Phone": "customer_phone",
    "Address": "customer_address",
    "Age": "customer_age",
    "Gender": "customer_gender",
    "Income": "customer_income",
    "Customer_Segment": "customer_segment"
})

In [ ]:
Extra_Columns

,customer_email,customer_phone,customer_address,customer_age,customer_gender,customer_income,customer_segment
0,Ebony39@gmail.com,1.414787e+09,3959 Amanda Burgs,21.0,Male,Low,Regular
1,Mark36@gmail.com,6.852900e+09,82072 Dawn Centers,19.0,Female,Low,Premium
2,Shane85@gmail.com,8.362160e+09,4133 Young Canyon,48.0,Male,Low,Regular
3,Mary34@gmail.com,2.776752e+09,8148 Thomas Creek Suite 100,56.0,Male,High,Premium
4,Charles30@gmail.com,9.098268e+09,5813 Lori Ports Suite 269,22.0,Male,Low,Premium
...,...,...,...,...,...,...,...
302005,Courtney60@gmail.com,7.466354e+09,389 Todd Path Apt. 159,31.0,Male,Medium,Regular
302006,Jennifer71@gmail.com,5.754305e+09,52809 Mark Forges,35.0,Female,Low,New
302007,Christopher100@gmail.com,9.382530e+09,407 Aaron Crossing Suite 495,41.0,Male,Low,Premium
302008,Rebecca65@gmail.com,9.373222e+09,3204 Baird Port,41.0,Male,Medium,New


In [ ]:
from faker import Faker
fake = Faker()

Extra_Columns["customer_password"] = [fake.password() for _ in range(len(Extra_Columns))]

In [ ]:
Extra_Columns = Extra_Columns.sample(n=len(df), random_state=42).reset_index(drop=True)

In [ ]:
Extra_Columns

,customer_email,customer_phone,customer_address,customer_age,customer_gender,customer_income,customer_segment,customer_password
0,David75@gmail.com,3.740098e+09,4618 Pamela Wells Suite 878,19.0,Female,Medium,Regular,@oCMNLmE+1
1,Adrian66@gmail.com,4.959731e+09,5158 Russell Creek Apt. 082,48.0,Male,Medium,New,^Si9KFrDsV
2,Robyn24@gmail.com,6.717169e+09,06198 Stephen Row,70.0,Male,Medium,Regular,Pq7qHLqu)G
3,Joshua53@gmail.com,6.929993e+09,491 Dennis Bridge Suite 461,24.0,Male,Medium,Regular,1Y6kPxhaW*
4,Paul89@gmail.com,2.665313e+09,61842 Anna Lodge Suite 608,22.0,Female,Medium,Regular,%tNdX#rIJ0
...,...,...,...,...,...,...,...,...
103672,Kimberly3@gmail.com,8.857652e+09,547 Gonzalez Route,68.0,Male,Low,Regular,b5yMRFsb_9
103673,James55@gmail.com,3.446319e+09,0983 Anthony Lodge Suite 951,21.0,Male,Medium,New,C()zIDkX_2
103674,Donald48@gmail.com,5.111705e+09,345 Elizabeth Point Suite 290,46.0,Male,Low,Regular,2dMpkdQP(7
103675,Kyle43@gmail.com,5.757558e+09,30005 Lyons Island Suite 537,20.0,Male,High,Regular,K(1(ZiQX*o


In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,1,credit_card,1,18.12,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,3,voucher,1,2.00,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2,voucher,1,18.59,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,1,boleto,1,141.46,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1,credit_card,3,179.12,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


In [ ]:
Enhanced_DF = pd.concat([df, Extra_Columns], axis=1)

In [ ]:
Enhanced_DF.shape

(103677, 30)

In [ ]:
Enhanced_DF

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_creation_date,review_answer_timestamp,customer_email,customer_phone,customer_address,customer_age,customer_gender,customer_income,customer_segment,customer_password
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2017-10-11 00:00:00,2017-10-12 03:43:48,David75@gmail.com,3.740098e+09,4618 Pamela Wells Suite 878,19.0,Female,Medium,Regular,@oCMNLmE+1
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2017-10-11 00:00:00,2017-10-12 03:43:48,Adrian66@gmail.com,4.959731e+09,5158 Russell Creek Apt. 082,48.0,Male,Medium,New,^Si9KFrDsV
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,2017-10-11 00:00:00,2017-10-12 03:43:48,Robyn24@gmail.com,6.717169e+09,06198 Stephen Row,70.0,Male,Medium,Regular,Pq7qHLqu)G
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,2018-08-08 00:00:00,2018-08-08 18:37:50,Joshua53@gmail.com,6.929993e+09,491 Dennis Bridge Suite 461,24.0,Male,Medium,Regular,1Y6kPxhaW*
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,2018-08-18 00:00:00,2018-08-22 19:07:58,Paul89@gmail.com,2.665313e+09,61842 Anna Lodge Suite 608,22.0,Female,Medium,Regular,%tNdX#rIJ0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103672,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,6359f309b166b0196dbf7ad2ac62bb5a,12209,...,2017-03-22 00:00:00,2017-03-23 11:02:08,Kimberly3@gmail.com,8.857652e+09,547 Gonzalez Route,68.0,Male,Low,Regular,b5yMRFsb_9
103673,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,da62f9e57a76d978d02ab5362c509660,11722,...,2018-03-01 00:00:00,2018-03-02 17:50:01,James55@gmail.com,3.446319e+09,0983 Anthony Lodge Suite 951,21.0,Male,Medium,New,C()zIDkX_2
103674,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,737520a9aad80b3fbbdad19b66b37b30,45920,...,2017-09-22 00:00:00,2017-09-22 23:10:57,Donald48@gmail.com,5.111705e+09,345 Elizabeth Point Suite 290,46.0,Male,Low,Regular,2dMpkdQP(7
103675,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,5097a5312c8b157bb7be58ae360ef43c,28685,...,2018-01-26 00:00:00,2018-01-27 09:16:56,Kyle43@gmail.com,5.757558e+09,30005 Lyons Island Suite 537,20.0,Male,High,Regular,K(1(ZiQX*o


In [ ]:
Enhanced_DF.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp', 'customer_email',
       'customer_phone', 'customer_address', 'customer_age', 'customer_gender',
       'customer_income', 'customer_segment', 'customer_password'],
      dtype='object')

# **1. Data Profiling**

In [ ]:
profile = ProfileReport(Enhanced_DF , title = "E-Commerce Profile")
profile.to_file("E-Commerce Profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 30/30 [00:21<00:00,  1.37it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# **2. Data Cleaning**

**Data Info**

In [ ]:
print("The Data Information:")
Enhanced_DF.info()

In [ ]:
print("Null Values:")
Enhanced_DF.isnull().sum()

In [ ]:
print("Total records before cleaning:", Enhanced_DF.shape[0])

**Some Logical Consistency Checks (For Safety Purposes)**

In [ ]:
Enhanced_DF = Enhanced_DF[~((Enhanced_DF["order_status"] == "delivered") &
          (Enhanced_DF["order_delivered_customer_date"].isna()))]

In [ ]:
Enhanced_DF.shape

**Get the dupliacted rows**

In [ ]:
duplicated_rows = Enhanced_DF[Enhanced_DF.duplicated()]
print(f"Number of duplicated rows: {duplicated_rows.shape[0]}")
display(duplicated_rows.head())

No duplicated values in the data.

**Clean phone numbers**

In [ ]:
Enhanced_DF["customer_phone"] = Enhanced_DF["customer_phone"].apply(lambda x: f"{int(x)}" if pd.notnull(x) else x)

In [ ]:
Enhanced_DF.shape

In [ ]:
Enhanced_DF["customer_phone"]

In [ ]:
Enhanced_DF["order_approved_at"] = Enhanced_DF["order_approved_at"].ffill()
Enhanced_DF["order_approved_at"] = Enhanced_DF["order_approved_at"].bfill()

In [ ]:
Enhanced_DF.shape

**Convert date columns**

In [ ]:
Date_Columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"]

In [ ]:
for Column in Date_Columns:
  Enhanced_DF[Column] = pd.to_datetime(Enhanced_DF[Column])

In [ ]:
Enhanced_DF.shape

In [ ]:
Enhanced_DF.head()

In [ ]:
dropped_cols = ["review_comment_title", "review_comment_message","customer_id"]
Enhanced_DF.drop(columns=dropped_cols, inplace=True, errors="ignore")
print("Null values after dropping them")
Enhanced_DF.isnull().sum()

In [ ]:
Enhanced_DF = Enhanced_DF[Enhanced_DF["payment_type"] != "not_defined"]

**Feature Engineering**

In [ ]:
Date_Columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]

for Column in Date_Columns:
    Enhanced_DF[Column] = pd.to_datetime(Enhanced_DF[Column])

In [ ]:
Enhanced_DF["delivery_time"] = (Enhanced_DF["order_delivered_customer_date"] - Enhanced_DF["order_purchase_timestamp"]).dt.days

**IQR Removing outliers for delivery_order in delivery_time**

In [ ]:
delivered = Enhanced_DF["order_status"] == "delivered"
Q1 = Enhanced_DF.loc[delivered, "delivery_time"].quantile(0.25)
Q3 = Enhanced_DF.loc[delivered, "delivery_time"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
Enhanced_DF = Enhanced_DF[~delivered | Enhanced_DF["delivery_time"].between(lower, upper)]

In [ ]:
Enhanced_DF.shape

(98527, 28)

**Final Check**

In [ ]:
print("\n")
print(f"The Final Shape: {Enhanced_DF.shape}")
print("The Remaining Missing Values:")
print("\n")
print(Enhanced_DF.isnull().sum())



The Final Shape: (98527, 28)
The Remaining Missing Values:


order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date     1856
order_delivered_customer_date    3019
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
payment_sequential                  0
payment_type                        0
payment_installments                0
payment_value                       0
review_id                           0
review_score                        0
review_creation_date                0
review_answer_timestamp             0
customer_email                    113
customer_phone                    119
customer_address                  101
customer_age                       49
customer_gender                   111
customer_income          

**Save new cleaned data**

In [ ]:
Enhanced_DF.to_csv("/content/cleaned_olist_data.csv", index=False)
print("\nCleaned dataset saved successfully.")


Cleaned dataset saved successfully.


In [ ]:
#save it on drive
from google.colab import drive
drive.mount('/content/drive')

Enhanced_DF.to_csv("/content/drive/MyDrive/cleaned_olist_data.csv", index=False)
print("Saved to Drive successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive successfully.


In [ ]:
print(Enhanced_DF.shape)

In [ ]:
Enhanced_DF.columns

# **3. Data Validation**

**Pandera: Schema Validation**

In [ ]:
Schema = DataFrameSchema(
    {
        "order_status" : pa.Column(pa.String, nullable = False),
        "payment_type" : pa.Column(pa.String, nullable = False),
        "payment_value" : pa.Column(pa.Float, Check.gt(0)),
        "review_score" : pa.Column(pa.Float, Check.in_range(1, 5), nullable=True),
        "delivery_time" : pa.Column(pa.Float, Check.ge(0), nullable = True),
        "customer_age": pa.Column(pa.Float, Check.between(18, 100), nullable=True),
        "order_purchase_timestamp" : pa.Column(pa.DateTime),
        "order_delivered_customer_date" : pa.Column(pa.DateTime, nullable = True),
    },
)

In [ ]:
try:
  Validated_df = Schema.validate(Enhanced_DF)
  print("The Pandera Validation Has Passed")
except Exception as e:
  print("The Pandera Validation Has Failed")
  print(e)

**Logical Validation: Business Rules**

In [ ]:
Invalid_Delivery = Enhanced_DF[(Enhanced_DF["order_status"] == "delivered") & (Enhanced_DF["order_delivered_customer_date"].isna())]
print(f"Invalid delivered data: {len(Invalid_Delivery)}")

In [ ]:
Invalid_Time = Enhanced_DF[(Enhanced_DF["delivery_time"].notna()) & (Enhanced_DF["delivery_time"] < 0)]  # To ensure that delivery times must be greater than or equal to 0.
print(f"Invalid delivery time data: {len(Invalid_Time)}")

In [ ]:
Invalid_Payment = Enhanced_DF[Enhanced_DF["payment_value"] <= 0]  # Only positive payment's allowed.
print(f"Invalid payment data: {len(Invalid_Payment)}")

In [ ]:
Enhanced_DF = Enhanced_DF[Enhanced_DF["payment_value"] > 0]  # To remove records with non-positive payment values
print(f"Remaining invalid payments: {len(Enhanced_DF[Enhanced_DF["payment_value"] <= 0])}")  # Verification

**Great Expectations: Validation Report**

In [ ]:
Context = gx.get_context()

In [ ]:
Data_Source = Context.data_sources.add_or_update_pandas(name = "my_pandas_datasource")

try:
    Data_Asset = Data_Source.add_dataframe_asset(name="my_df_asset")
except ValueError:
    Data_Asset = Data_Source.get_asset("my_df_asset")

Batch_Request = Data_Asset.build_batch_request(options={"dataframe": Enhanced_DF})

My_Validator = Context.get_validator(batch_request = Batch_Request, dataframe = Enhanced_DF)

In [ ]:
My_Validator.expect_column_values_to_not_be_null("order_status")
My_Validator.expect_column_values_to_be_between("payment_value", min_value = 0)
My_Validator.expect_column_values_to_be_between("customer_age", min_value = 18, max_value = 100)
My_Validator.expect_column_values_to_be_between("review_score", min_value = 1, max_value = 5)
My_Validator.expect_column_values_to_be_in_set("customer_gender", ["Male", "Female"])
My_Validator.expect_column_values_to_be_in_set("order_status", ["delivered", "shipped", "canceled", "processing", "invoiced", "unavailable", "approved", "created"])

In [ ]:
The_Results = My_Validator.validate()

In [ ]:
print(The_Results)

**Cerberus: Rule-Based Validation**

In [ ]:
Cerberus_Schema = {
    "payment_value" : {"type" : "float", "min" : 0},
    "customer_age": {"type": "float", "min": 18},
    "review_score" : {"type" : "float", "min" : 1, "max" : 5},
    "order_status" : {"type" : "string"},
    "customer_segment": {"type": "string", "allowed": ["Regular", "Premium", "New", "VIP"]},
}

V = Validator(Cerberus_Schema, allow_unknown=True)  # It won't complain about other columns with allow_unknown=True

def Row_Checker(Row):              # To get the mask of valid rows
  return V.validate(Row.to_dict())

Validity_Check = Enhanced_DF.apply(Row_Checker, axis = 1)  # To apply the check

if Validity_Check.all():
  print("Complete pass for all of the records regarding Cerberus Validation.")
else:
  Number_Of_Errors = (~Validity_Check).sum()
  print(f"Cerberus found {Number_Of_Errors} errors (invalid records).")
  print("Example invalid rows:")
  print(Enhanced_DF[~Validity_Check].head(5))

Cerberus found 76 errors (invalid records).
Example invalid rows:
                               order_id order_status order_purchase_timestamp  \
649    ba863950726dd067d21b2262ab48df78    delivered      2017-10-22 14:25:40   
681    e6c3435aca88a50c893f06b6b13e9242    delivered      2017-06-19 18:00:01   
2340   0b192907ec8563bda5210247497aa9af    delivered      2017-10-24 02:04:42   
10018  076e07cd8d9c8b47dfc2fbb22c58e895    delivered      2017-10-18 09:29:12   
12525  d5bd25a973bf9ea4a5fb210e058ede88    delivered      2018-06-01 18:39:25   

        order_approved_at order_delivered_carrier_date  \
649   2017-10-22 14:49:15          2017-10-23 18:50:12   
681   2017-06-19 18:10:19          2017-06-20 15:47:27   
2340  2017-10-24 02:25:37          2017-10-25 18:23:33   
10018 2017-10-18 09:49:14          2017-10-18 16:40:22   
12525 2018-06-05 04:52:00          2018-06-05 15:03:00   

      order_delivered_customer_date order_estimated_delivery_date  \
649             2017-10-27 21